In [29]:
import numpy as np
import cv2
from PIL import Image
import os
from pathlib import Path

In [30]:
TILES_DIR = "./fu/tile_imgs"
STRIPS_DIR = "./fu/type_imgs"

In [31]:
# clear output
def clear_tile_folders():
    base = Path(TILES_DIR)
    
    # Check if the base directory exists
    if not base.exists():
        print(f"Error: The directory {TILES_DIR} does not exist.")
        return

    # Iterate through every item in the base directory
    for subfolder in base.iterdir():
        # Only process if it is one of your 5 specific subdirectories
        if subfolder.is_dir():
            print(f"Clearing contents of: {subfolder.name}")
            
            # Iterate through all files inside the subfolder
            for file in subfolder.iterdir():
                try:
                    if file.is_file():
                        file.unlink()  # Deletes the file
                except Exception as e:
                    print(f"Could not delete {file.name}: {e}")

clear_tile_folders()

Clearing contents of: d
Clearing contents of: m
Clearing contents of: p
Clearing contents of: s
Clearing contents of: z


In [32]:
class MahjongTileDetector:
    def __init__(self, threshold=50):
        self.threshold = threshold

    def extract_and_standardize_tiles(self, input_image_path, expected_count, output_directory, prefix):
        print(f"\n--- Processing: {input_image_path} ---")
        
        img = cv2.imread(input_image_path, cv2.IMREAD_UNCHANGED)
        if img is None: return

        # 1. Create a binary mask
        if img.shape[2] == 4:
            gray = img[:, :, 3] 
        else:
            hsv = cv2.cvtColor(img[:,:,:3], cv2.COLOR_BGR2HSV)
            _, _, gray = cv2.split(hsv)
        
        _, binary = cv2.threshold(gray, self.threshold, 255, cv2.THRESH_BINARY)

        # --- FIX: Break bridges between touching tiles ---
        # A 3x3 or 5x5 kernel usually works well for horizontal gaps
        kernel = np.ones((3, 3), np.uint8)
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)

        # 2. Vertical Projection
        projection = np.sum(binary, axis=0)
        
        # 3. Dynamic Splitting
        # Instead of val == 0, we look for values below a small percentage of the max
        # This handles cases where the "gap" isn't perfectly black
        cut_threshold = np.max(projection) * 0.1  
        
        tile_spans = []
        in_tile = False
        start_x = 0
        
        for x, val in enumerate(projection):
            if val > cut_threshold and not in_tile:
                start_x = x
                in_tile = True
            elif val <= cut_threshold and in_tile:
                # Minimum width check (e.g. 5px) to ignore tiny noise
                if (x - start_x) > 5:
                    tile_spans.append((start_x, x))
                in_tile = False
        if in_tile: tile_spans.append((start_x, len(projection)))

        # 4. Enforce the expected count
        # If we found too many, take the widest ones. 
        # If we found only 1, the cut_threshold is likely too low.
        tile_spans = sorted(tile_spans, key=lambda s: s[1]-s[0], reverse=True)[:expected_count]
        tile_spans.sort()

        if not os.path.exists(output_directory):
            os.makedirs(output_directory)

        if len(tile_spans) <= 1:
            print(f"⚠️ Warning: Only detected {len(tile_spans)} tile(s). Try lowering the 'threshold' value (currently {self.threshold}).")

        for i, (x1, x2) in enumerate(tile_spans):
            tile_crop = img[:, x1:x2]
            
            # Vertical auto-crop
            tile_gray = binary[:, x1:x2]
            coords = cv2.findNonZero(tile_gray)
            if coords is not None:
                _, ty, _, th = cv2.boundingRect(coords)
                tile_crop = tile_crop[ty:ty+th, :]

            tile_name = f"{prefix}{i+1}.png"
            if tile_crop.shape[2] == 4:
                final_img = Image.fromarray(cv2.cvtColor(tile_crop, cv2.COLOR_BGRA2RGBA))
            else:
                final_img = Image.fromarray(cv2.cvtColor(tile_crop, cv2.COLOR_BGR2RGB))
            
            final_img = final_img.resize((60,95))
            final_img.save(os.path.join(output_directory, tile_name))
        print(f"Done: Saved {len(tile_spans)} tiles.")

# --- EXECUTION ---



target_detector = MahjongTileDetector()

# 1. Winds (image_0.png) - 4 tiles: Ton, Nan, Sha, Pei
target_detector.extract_and_standardize_tiles(
    os.path.join(STRIPS_DIR,"f.JPG"), expected_count=4, output_directory=os.path.join(TILES_DIR,"z"), prefix='z')

# 2. Manzu (image_1.png) - 9 tiles: 1m to 9m
target_detector.extract_and_standardize_tiles(
    os.path.join(STRIPS_DIR,"m.JPG"), expected_count=9, output_directory=os.path.join(TILES_DIR,"m"), prefix='m')

# 3. Pinzu (image_2.png) - 9 tiles: 1p to 9p
target_detector.extract_and_standardize_tiles(
    os.path.join(STRIPS_DIR,"p.JPG"), expected_count=9, output_directory=os.path.join(TILES_DIR,"p"), prefix='p')

# 4. Souzu (image_3.png) - 9 tiles: 1s to 9s
target_detector.extract_and_standardize_tiles(
    os.path.join(STRIPS_DIR,"s.JPG"), expected_count=9, output_directory=os.path.join(TILES_DIR,"s"), prefix='s')

# 5. Dragons (image_4.png) - 3 tiles: Haku, Hatsuu, Chun
target_detector.extract_and_standardize_tiles(
    os.path.join(STRIPS_DIR,"sn.JPG"), expected_count=3, output_directory=os.path.join(TILES_DIR,"d"), prefix='d')

print(f"\n✅ All CV-detected, standardized assets saved to: {TILES_DIR}")


--- Processing: ./fu/type_imgs\f.JPG ---
Done: Saved 4 tiles.

--- Processing: ./fu/type_imgs\m.JPG ---
Done: Saved 9 tiles.

--- Processing: ./fu/type_imgs\p.JPG ---
Done: Saved 9 tiles.

--- Processing: ./fu/type_imgs\s.JPG ---
Done: Saved 9 tiles.

--- Processing: ./fu/type_imgs\sn.JPG ---
Done: Saved 3 tiles.

✅ All CV-detected, standardized assets saved to: ./fu/tile_imgs
